# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [ ]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection/"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml

In [ ]:
# Step 2: Install Required Dependencies
# Install packages needed for document processing and text chunking

%pip install docling markdown-it-py
%pip install --upgrade transformers

In [ ]:
# Step 3: Load Processed Documents
import glob
import os
from pathlib import Path

# In our example above docling step produces markdown of all the PDF files in document_collection
md_files = sorted(glob.glob(f"{data_dir}/*.md"))

if not md_files:
    raise ValueError(f"No markdown files found in {data_dir}. Run Step 1 first.")

documents = []
for md_path in md_files:
    with open(md_path, "r", encoding="utf-8") as f:
        documents.append(
            {
                "source_file": os.path.basename(md_path),
                "document_outline": Path(md_path).stem,
                "text": f.read(),
            }
        )

print(f"Loaded {len(documents)} markdown files from {data_dir}")
for doc in documents:
    print(f"  - {doc['source_file']}")


In [ ]:
# Step 4: Text Chunking and Dataset Creation

from collections import Counter
import re
from markdown_it import MarkdownIt
from typing import List
import datasets


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """
    Splits Markdown text into chunks at block-level elements
    (headings, paragraphs, lists, tables, code, blockquotes).
    Adds overlap (in words) between all consecutive chunks.

    Args:
        text: The markdown text to be chunked
        max_tokens: Maximum number of words per chunk
        overlap: Number of overlapping words between consecutive chunks

    Returns:
        List of text chunks with specified overlap
    """

    md = MarkdownIt()
    tokens = md.parse(text)

    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                chunks.append(" ".join(current_words))
                current_words = current_words[-overlap:] if overlap > 0 else []

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


def extract_headings(markdown_text: str, max_headings: int = 3) -> List[str]:
    headings = []
    for line in markdown_text.splitlines():
        line = line.strip()
        if not line.startswith("#"):
            continue
        heading = re.sub(r"^#+\s*", "", line).strip()
        if heading:
            headings.append(heading)
        if len(headings) >= max_headings:
            break
    return headings


def extract_keywords(markdown_text: str, max_keywords: int = 3) -> List[str]:
    stopwords = {
        "about", "above", "after", "again", "against", "among", "an", "and",
        "are", "as", "at", "be", "because", "been", "before", "being", "below",
        "between", "both", "but", "by", "can", "com", "como", "como", "com",
        "da", "das", "de", "del", "desde", "do", "dos", "during", "each", "for",
        "from", "had", "has", "have", "in", "into", "is", "it", "its", "mais",
        "na", "nas", "no", "nos", "of", "on", "or", "os", "para", "por", "que",
        "sem", "ser", "seu", "seus", "sua", "suas", "the", "their", "them", "then",
        "there", "these", "this", "those", "to", "um", "uma", "umas", "uns", "with",
    }

    tokens = re.findall(r"[A-Za-z0-9]{4,}", markdown_text.lower())
    tokens = [t for t in tokens if t not in stopwords and not t.isdigit()]
    most_common = [term for term, _ in Counter(tokens).most_common(max_keywords)]
    return most_common


def build_icl_queries(document_outline: str, headings: List[str], keywords: List[str]) -> tuple[str, str, str]:
    h1 = headings[0] if len(headings) > 0 else "tema principal"
    h2 = headings[1] if len(headings) > 1 else (keywords[0] if len(keywords) > 0 else "conceitos centrais")
    h3 = headings[2] if len(headings) > 2 else (keywords[1] if len(keywords) > 1 else "implicacoes praticas")

    q1 = f"No documento '{document_outline}', quais sao os pontos principais da secao '{h1}'?"
    q2 = f"Como o conteudo de '{h2}' se conecta ao objetivo geral de '{document_outline}'?"
    q3 = f"Quais aplicacoes, riscos ou decisoes praticas podem ser derivadas de '{h3}' neste documento?"

    return q1, q2, q3


all_rows = []
doc_to_icl = {}
for doc in documents:
    doc_chunks = chunk_markdown(doc["text"], max_tokens=5000, overlap=1000)
    if not doc_chunks:
        continue

    headings = extract_headings(doc["text"])
    keywords = extract_keywords(doc["text"])
    q1, q2, q3 = build_icl_queries(doc["document_outline"], headings, keywords)

    icl_document = " ".join(doc_chunks[0].split()[:250])
    doc_to_icl[doc["source_file"]] = {
        "icl_document": icl_document,
        "icl_query_1": q1,
        "icl_query_2": q2,
        "icl_query_3": q3,
    }

    for chunk in doc_chunks:
        all_rows.append(
            {
                "document": chunk,
                "document_outline": doc["document_outline"],
                "source_file": doc["source_file"],
            }
        )

if not all_rows:
    raise ValueError("No chunks were produced from markdown documents.")

seed_data = datasets.Dataset.from_list(all_rows)

domain = os.getenv("SEED_DOMAIN", "general")


def add_icl_fields(sample):
    doc_icl = doc_to_icl[sample["source_file"]]
    return {
        "icl_document": doc_icl["icl_document"],
        "icl_query_1": doc_icl["icl_query_1"],
        "icl_query_2": doc_icl["icl_query_2"],
        "icl_query_3": doc_icl["icl_query_3"],
        "domain": domain,
    }


seed_data = seed_data.map(add_icl_fields)

seed_data_path = os.getenv("SEED_DATA_PATH", "seed_data.jsonl")
seed_data.to_json(seed_data_path, orient="records", lines=True)

print(f"Saved seed data to: {seed_data_path}")
print(f"Total chunks: {len(seed_data)}")
print(f"Columns: {seed_data.column_names}")
print("Example generated queries:")
for source_file, icl_values in list(doc_to_icl.items())[:3]:
    print(f"- {source_file}")
    print(f"  Q1: {icl_values['icl_query_1']}")
    print(f"  Q2: {icl_values['icl_query_2']}")
    print(f"  Q3: {icl_values['icl_query_3']}")


### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook